In [1]:
import pandas as pd
import logging
import sys
from pathlib import Path

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)

COMPLETED_LIST_DIR = Path("completed_lists")

In [2]:
import os 
import errno

def read_completed_list(list_path, create=True) -> list:
    """
    Reads a list from a file.

    Args:
        list_path (str): The path to the file.
        create (bool, optional): If True, creates the file if it doesn't exist. Defaults to True.

    Returns:
        list: The list read from the file.

    Raises:
        FileNotFoundError: If the file doesn't exist and create is False.
    """
    # Check if path exists
    if not os.path.exists(list_path):
        if create:
            open(list_path, 'a').close()
            return []
        else:
            raise FileNotFoundError(errno.ENOENT, os.strerror(errno.ENOENT), list_path)

    # If so, open and return the list
    with open(list_path, "r") as f:
        return [line.rstrip('\n') for line in f]

def write_completed_list(list_path, completed_list):
    """
    Writes a list to a file.

    Args:
        list_path (str): The path to the file.
        completed_list (list): The list to be written to the file.
    """
    with open(list_path, "a") as f:
        for item in completed_list:
            f.write(item + '\n')


In [3]:
list_names = [
    "thermo",
    "merged_omni_indices",
    "merged_omni_solar_wind",
    "merged_omni_magnetic_field",
    "goes_256nm_sw",
    "goes_284nm_sw",
    "goes_304nm_sw",
    "goes_1175nm_sw",
    "goes_1216nm_sw",
    "goes_1335nm_sw",
    "goes_1405nm_sw",
    "soho_data",
    "nrlmsise00_time_series"
]

# Create an empty dictionary to store completed lists
completed_lists = {}

# Iterate over each name in the list_names list
for name in list_names:
    # Read the completed list file for the current name and store it in the completed_lists dictionary
    completed_lists[name] = read_completed_list(Path("completed_lists") / (name + ".txt")) 

In [4]:
import glob

BASE_PATH = Path("/mnt/")
merge_data_paths = {}
for name in list_names:
    match name:
        case "thermo":
            merge_data_paths[name] = glob.glob(str(BASE_PATH / "satellite-data-processed/tudelft/*/*/*.parquet"))
        case "merged_omni_indices":
            merge_data_paths[name] = glob.glob(str(BASE_PATH / "physical-drivers-processed/OMNIWEB/*/indices_*.parquet"))
        case "merged_omni_solar_wind":
            merge_data_paths[name] = glob.glob(str(BASE_PATH / "physical-drivers-processed/OMNIWEB/*/solar_wind_velocity_*.parquet"))
        case "merged_omni_magnetic_field":
            merge_data_paths[name] = glob.glob(str(BASE_PATH / "physical-drivers-processed/OMNIWEB/*/magnetic_*.parquet"))
        # GOES
        case "soho_data":
            merge_data_paths[name] = glob.glob(str(BASE_PATH / "physical-drivers-processed/SOHO/*/*.parquet"))
        # MSIS

In [5]:
import fastparquet

def write_chunk(path, data):
    """
    Write data to a Parquet file at the given path.

    If the file already exists, the data will be appended to it.
    If the file does not exist, a new Parquet file will be created.

    Parameters:
    - path (str): The path to the Parquet file.
    - data: The data to be written to the file.

    Returns:
    None
    """
    if os.path.exists(path):
        fastparquet.write(path, data, append=True)
    else:
        concatenated_data.to_parquet(path, engine="fastparquet")


for name in list_names:
    if "goes" in name:
        continue

    # Check if there is data
    if len(merge_data_paths[name]) == 0:
        logging.warn(f"No files found for {name}, skipping...")
        continue
    
    # Grab the unprocessed changes
    unprocessed_list = [new for new in merge_data_paths[name] if new not in completed_lists[name]]
    if len(unprocessed_list) == 0:
        logging.warn(f"No new data found for {name}, skipping...")
        continue

    logging.info(f"Processing new {len(unprocessed_list)} files for {name}.")
    existing_path = Path("/mnt/asimovs-accumulated-data/"+name+".parquet")
    
    # Chunked processing of files, rarely needed if using at daily cadance but will be if at greater
    # This script is intended to be run a low-cost VM
    c = 0
    CHUNK_SIZE = 100
    total_unprocessed = len(unprocessed_list)
    while c < total_unprocessed//CHUNK_SIZE + 1:
        a = c*CHUNK_SIZE
        b = (c+1)*CHUNK_SIZE
        b = b if not b > total_unprocessed else total_unprocessed
        
        # exactly on the edge
        if a == b:
            break

        concatenated_data = pd.concat([pd.read_parquet(file_path, engine="fastparquet") for file_path in unprocessed_list[a:b]])
        c += 1
        logging.info(f"Writing files {a}-{b}, {(b/total_unprocessed)*100:.02f}%, and updating completed cache.")
        write_chunk(existing_path, concatenated_data)
        write_completed_list(str(COMPLETED_LIST_DIR) + "/" + name + ".txt", unprocessed_list[a:b])

    del concatenated_data

INFO:root:Processing new 1206 files for thermo.
DEBUG:parquet:Using json encoder/decoder
INFO:root:Writing files 0-100, 8.29%, and updating completed cache.
INFO:root:Writing files 100-200, 16.58%, and updating completed cache.
INFO:root:Writing files 200-300, 24.88%, and updating completed cache.
INFO:root:Writing files 300-400, 33.17%, and updating completed cache.
INFO:root:Writing files 400-500, 41.46%, and updating completed cache.
INFO:root:Writing files 500-600, 49.75%, and updating completed cache.
INFO:root:Writing files 600-700, 58.04%, and updating completed cache.
INFO:root:Writing files 700-800, 66.33%, and updating completed cache.
INFO:root:Writing files 800-900, 74.63%, and updating completed cache.
INFO:root:Writing files 900-1000, 82.92%, and updating completed cache.
INFO:root:Writing files 1000-1100, 91.21%, and updating completed cache.
INFO:root:Writing files 1100-1200, 99.50%, and updating completed cache.
INFO:root:Writing files 1200-1206, 100.00%, and updating 

ValueError: No objects to concatenate

In [46]:
len(unprocessed_list)

1204

In [48]:
write_completed_list(str(COMPLETED_LIST_DIR/(name+".txt")), unprocessed_list)

In [ ]:
df = pd.read_parquet("/mnt/satellite-data-processed/tudelft/version_02/CHAMP_data/db_CH_DNS_ACC_2000-07_v02.parquet")

In [ ]:
df

,all__dates_datetime__,tudelft_thermo__altitude__[m],tudelft_thermo__longitude__[deg],tudelft_thermo__latitude__[deg],tudelft_thermo__ground_truth_thermospheric_density__[kg/m**3],tudelft_thermo__satellite__,all__year__[y],all__day_of_year__[d],all__seconds_in_day__[s],space_environment_technologies__f107_obs__,space_environment_technologies__f107_average__,space_environment_technologies__s107_obs__,space_environment_technologies__s107_average__,space_environment_technologies__m107_obs__,space_environment_technologies__m107_average__,space_environment_technologies__y107_obs__,space_environment_technologies__y107_average__,JB08__d_st_dt__[K],celestrack__ap_average__
0,2000-07-29 00:59:47,471231.023,1.108932,39.878609,1.163238e-12,champ,2000,211,3587.0,157.8,181.3,195.5,195.0,167.9,183.8,181.8,167.4,105.0,27.0
1,2000-07-29 00:59:57,470866.299,1.117947,39.240399,1.185663e-12,champ,2000,211,3597.0,157.8,181.3,195.5,195.0,167.9,183.8,181.8,167.4,105.0,27.0
2,2000-07-29 01:00:07,470500.961,1.126055,38.602045,1.185089e-12,champ,2000,211,3607.0,157.8,181.3,195.5,195.0,167.9,183.8,181.8,167.4,105.0,27.0
3,2000-07-29 01:00:17,470135.128,1.133291,37.963550,1.177544e-12,champ,2000,211,3617.0,157.8,181.3,195.5,195.0,167.9,183.8,181.8,167.4,105.0,27.0
4,2000-07-29 01:00:27,469768.919,1.139690,37.324913,1.166322e-12,champ,2000,211,3627.0,157.8,181.3,195.5,195.0,167.9,183.8,181.8,167.4,105.0,27.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22018,2000-07-31 23:58:57,451613.934,-167.834361,-41.514304,2.705666e-12,champ,2000,213,86337.0,149.9,181.0,188.7,195.2,164.7,183.5,172.1,167.2,50.0,21.0
22019,2000-07-31 23:59:07,451611.298,-167.822532,-40.872650,2.706122e-12,champ,2000,213,86347.0,149.9,181.0,188.7,195.2,164.7,183.5,172.1,167.2,50.0,21.0
22020,2000-07-31 23:59:17,451610.287,-167.811729,-40.230916,2.711298e-12,champ,2000,213,86357.0,149.9,181.0,188.7,195.2,164.7,183.5,172.1,167.2,50.0,21.0
22021,2000-07-31 23:59:27,451610.976,-167.801910,-39.589103,2.739367e-12,champ,2000,213,86367.0,149.9,181.0,188.7,195.2,164.7,183.5,172.1,167.2,50.0,21.0
